# Validation of Low Pileup Pilot


In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import uproot
import sys
import time
from tqdm import tqdm
from pathlib import Path
import atlasify as atl
atl.ATLAS = "ColliderML"

sys.path.append("../")
from edm4hep_utils import load_edm4hep_file

## Roadmap

1. Load ~10 EDM4Hep files from both ttbar and chichi
2. For each, plot number of tracker hits and calo hits (in each subdetector)
3. Plot total energy depositions in the calo (and look at MET)
4. Check that each event is unique
5. Check if we can get the vertices and differentiate hard scatter and pileup
6. Check if we can see the SUSY and top processes

## Debug Pythia

In [12]:
particles_file = "/global/cfs/projectdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/pda_tests/pda_testing_susy1/particles.root"
particles = uproot.open(particles_file)["particles"]

In [14]:
particles.show()
particles.arrays()


name                 | typename                 | interpretation                
---------------------+--------------------------+-------------------------------
event_id             | uint32_t                 | AsDtype('>u4')
particle_id          | std::vector<uint64_t>    | AsJagged(AsDtype('>u8'), he...
particle_type        | std::vector<int32_t>     | AsJagged(AsDtype('>i4'), he...
process              | std::vector<uint32_t>    | AsJagged(AsDtype('>u4'), he...
vx                   | std::vector<float>       | AsJagged(AsDtype('>f4'), he...
vy                   | std::vector<float>       | AsJagged(AsDtype('>f4'), he...
vz                   | std::vector<float>       | AsJagged(AsDtype('>f4'), he...
vt                   | std::vector<float>       | AsJagged(AsDtype('>f4'), he...
px                   | std::vector<float>       | AsJagged(AsDtype('>f4'), he...
py                   | std::vector<float>       | AsJagged(AsDtype('>f4'), he...
pz                   | std::vector<float>   

<Array [{event_id: 0, particle_id: [], ...}] type='1 * {event_id: uint32, p...'>

name                 | typename                 | interpretation                
---------------------+--------------------------+-------------------------------
event_id             | uint32_t                 | AsDtype('>u4')                


In [11]:
uproot.open(particles_file).keys()

['particles;1']

In [7]:
particles

<Array [{event_id: 0, particle_id: [], ...}] type='1 * {event_id: uint32, p...'>

In [9]:
# Particles to dataframe
particles["particle_type"][0]

<Array [] type='0 * int32'>

In [3]:
particles

AttributeError: 'Model_TTree_v20' object has no attribute 'array'

### Load in edm4hep file

In [2]:
ttbar_file = "/global/cfs/cdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/low_pileup_pilot/gg2ttbar/v1/runs/0/edm4hep.root"
ttbar_events = load_edm4hep_file(ttbar_file)

chichi_file = "/global/cfs/cdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/low_pileup_pilot/chichi/v1/runs/0/edm4hep.root"
chichi_events = load_edm4hep_file(chichi_file)

In [ ]:
ttbar_events.keys()

In [ ]:
ttbar_hits, ttbar_particles, ttbar_cells = ttbar_events["hits"], ttbar_events["particles"], ttbar_events["cells"]

chichi_hits, chichi_particles, chichi_cells = chichi_events["hits"], chichi_events["particles"], chichi_events["cells"]


## High-level study

### Initial visualizations